In [51]:
import gymnasium as gym
env = gym.make("ALE/Breakout-v5")

In [52]:
import cv2
import numpy as np
from typing import Tuple
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from typing import Optional
import time
import pandas as pd
import matplotlib.pyplot as plt

ATARI ENVIRONMENT SETUP :

Image Preprocessing Functions

In [53]:
def rgb_to_grayscale(frame: np.ndarray) -> np.ndarray:
    """Converts RGB frame to grayscale using luminosity method"""
    gray = np.dot(frame[..., :3], [0.2989, 0.5870, 0.1140])
    return gray.astype(np.uint8)

def resize_frame(frame: np.ndarray, target_size: Tuple[int, int] = (80, 80)) -> np.ndarray:
    """Resizes frame using cv2.INTER_AREA interpolation"""
    return cv2.resize(frame, target_size, interpolation=cv2.INTER_AREA)

def normalize_frame(frame: np.ndarray) -> np.ndarray:
    """Normalizes pixel values to [0.0, 1.0] range"""
    return frame.astype(np.float32) / 255.0

def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    """Chains grayscale, resize, and normalize operations"""
    gray = rgb_to_grayscale(frame)
    resized = resize_frame(gray)
    normalized = normalize_frame(resized)
    return normalized

Frame Stacking Implementation

In [54]:
class FrameStack:
    def __init__(self, maxlen: int = 4):
        """Initialize deque with maxlen capacity"""
        self.maxlen = maxlen
        self.frames = deque(maxlen=maxlen)

    def push(self, frame: np.ndarray) -> None:
        """Add preprocessed frame to deque"""
        self.frames.append(frame)

    def get_stack(self) -> np.ndarray:
        """Return stacked frames as (maxlen, height, width)"""
        if len(self.frames) < self.maxlen:
            h, w = self.frames[0].shape if self.frames else (80, 80)
            missing = self.maxlen - len(self.frames)
            pad = [np.zeros((h, w), dtype=np.float32)] * missing
            stacked = list(pad) + list(self.frames)
        else:
            stacked = list(self.frames)

        return np.stack(stacked, axis=0)

    def reset(self) -> None:
        """Clear the deque"""
        self.frames.clear()

Atari Environment Wrapper

In [55]:
class AtariWrapper:
    def __init__(self, env_name: str = "BreakoutNoFrameskip-v4", frame_skip: int = 4):
        """Initialize environment and frame stack"""
        self.env = gym.make(env_name, render_mode=None)
        self.frame_stack = FrameStack(maxlen=4)
        self.frame_skip = frame_skip

        # Action mapping for Breakout
        self.action_meanings = self.env.unwrapped.get_action_meanings()
        self.action_map = {
            0: 0,  # NOOP
            1: 1,  # FIRE
            2: 2,  # RIGHT
            3: 3   # LEFT 
        }

    def reset(self) -> np.ndarray:
        """Reset environment and return initial stacked state"""
        obs, _ = self.env.reset()
        self.frame_stack.reset()
        processed = preprocess_frame(obs)
        for _ in range(self.frame_stack.maxlen):
            self.frame_stack.push(processed)
        return self.get_state()

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, dict]:
        """Execute action with frame skipping"""
        mapped_action = self.action_map.get(action, 0)
        total_reward = 0.0
        done = False
        info = {}

        for _ in range(self.frame_skip):
            obs, reward, terminated, truncated, info = self.env.step(mapped_action)
            processed = preprocess_frame(obs)
            self.frame_stack.push(processed)
            total_reward += reward
            done = terminated or truncated
            if done:
                break

        return self.get_state(), total_reward, done, info

    def get_state(self) -> np.ndarray:
        """Return current stacked state"""
        return self.frame_stack.get_stack() 

    def close(self):
        """Close the environment"""
        self.env.close()

PRIORITISED EXPERIENCE REPLAY :

SumTree Data Structure

In [56]:
class SumTree:
    def __init__(self, capacity: int):
        """Initialize tree structure for efficient priority sampling"""
        # Setup binary tree arrays and tracking variables
        self.capacity=capacity
        self.tree=np.zeros(2*capacity-1,dtype=np.float32)
        self.data=np.zeros(capacity,dtype=object)
        self.data_pointer=0

    def _propagate(self, idx: int, change: float) -> None:
        """Propagate priority changes up the tree"""
        # Update parent nodes recursively to root
        idx=(idx-1)//2
        while(idx>=0):
            self.tree[idx]+=change
            idx=(idx-1)//2

    def _retrieve(self, idx: int, s: float) -> int:
        """Retrieve leaf index for given priority value"""
        # Navigate tree based on cumulative priorities
        while True:
            left_child=2*idx+1
            right_child=left_child+1
            if left_child>=len(self.tree):
                leaf_idx=idx
                break
            else:
                if s<=self.tree[left_child]:
                    idx=left_child
                else:
                    s=s-self.tree[left_child]
                    idx=right_child
        return leaf_idx


    def total_priority(self) -> float:
        """Return total priority (root value)"""
        return self.tree[0]

    def add(self, priority: float, data: object) -> None:
        """Store experience with priority"""
        # Add data to tree and propagate changes
        tree_idx=self.data_pointer+self.capacity-1
        self.update(tree_idx,priority)
        self.data[self.data_pointer]=data
        self.data_pointer+=1
        if self.data_pointer>=self.capacity:
            self.data_pointer=0

    def update(self, idx: int, priority: float) -> None:
        """Update priority of existing experience"""
        # Update node and propagate change
        change=priority-self.tree[idx]
        self.tree[idx]=priority
        self._propagate(idx,change)

    def get_leaf(self, s: float) -> tuple[int, float, object]:
        """Sample leaf based on priority value s"""
        # Find leaf and return index, priority, data
        leaf_idx=self._retrieve(0,s)
        data_idx=leaf_idx-self.capacity+1
        return leaf_idx,self.tree[leaf_idx],self.data[data_idx]

Prioritized Replay Buffer

In [57]:
class PrioritizedReplayBuffer:
    def __init__(self, capacity: int, alpha: float = 0.6, beta_start: float = 0.4):
        """Initialize prioritized replay buffer"""
        # Set up the SumTree and store prioritization parameters.
        # Initialize beta annealing schedule and numerical stability constants.
        # Set up frame counting for importance sampling weight calculation.
        self.capacity=capacity
        self.alpha=alpha
        self.beta_start=beta_start
        self.frame_idx=0
        self.frames=1000000
        self.sum_tree=SumTree(capacity)
        self.epsilon=1e-3
        self.size=0

    def _get_beta(self) -> float:
        """Calculate current beta value (anneals to 1.0)"""
        # Implement linear annealing from beta_start to 1.0 over training.
        self.frame_idx+=1
        update_beta=self.beta_start+self.frame_idx*(1-self.beta_start)/self.frames
        return min(1.0,update_beta)

    def push(self, state, action, reward, next_state, done) -> None:
        """Store experience with maximum priority"""
        # Package the experience tuple and assign appropriate priority.
        # Use maximum existing priority for new experiences to ensure sampling
        experience=(state,action,reward,next_state,done)
        self.size+=1
        priority=np.max(self.sum_tree.tree[(self.sum_tree.capacity-1):])
        if priority==0:
            priority=1
        self.sum_tree.add(priority,experience)


    def sample(self, batch_size: int) -> tuple[list, np.ndarray, np.ndarray]:
        """Sample batch with importance sampling weights"""
        # Divide priority range into segments for stratified sampling.
        # Calculate importance sampling weights to correct for sampling bias.
        # Return experiences, tree indices, and normalized weights.
        sample=[]
        indices=[]
        experiences=[]
        total_priority=self.sum_tree.total_priority()
        segment=total_priority/batch_size
        for i in range(batch_size):
            a=segment*i
            b=segment*(i+1)
            s=np.random.uniform(a,b)
            leaf_idx,priority,data=self.sum_tree.get_leaf(s)
            sample.append(priority)
            indices.append(leaf_idx)
            experiences.append(data)
        indices=np.array(indices)
        temp=np.array(sample)/total_priority
        weights=((1/self.size)*(1/temp))**self._get_beta()
        normalized=weights/weights.max()
        return experiences,indices,normalized


    def update_priorities(self, indices, td_errors) -> None:
        """Update priorities based on TD errors"""
        # Convert TD errors to priorities using alpha exponent.
        # Add small epsilon for numerical stability.
        # Update tree nodes with new priority values.
        for idx, error in zip(indices,td_errors):
            priority=(np.abs(error)+self.epsilon)**self.alpha
            self.sum_tree.update(idx,priority)

    def __len__(self) -> int:
        """Return current buffer size"""
        # Return the number of experiences currently stored.
        return self.size

DEEP Q-NETWORK (DQN) ARCHITECTURE

CNN Architecture Design

In [ ]:
# class DQN(nn.Module):
#     def __init__(self, input_dim: tuple[int, int, int], action_dim: int):
#         super(DQN, self).__init__()
#         self.conv1 = nn.Conv2d(in_channels=input_dim[0], out_channels=16, kernel_size=8, stride=4)
#         self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=4, stride=2)
#         self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1)

#         conv_out_size = self._get_conv_out_size(input_dim)
#         self.value_stream = nn.Sequential(nn.Linear(conv_out_size, 256),nn.ReLU(),nn.Linear(256, 1))

#         # Advantage stream
#         self.advantage_stream = nn.Sequential(nn.Linear(conv_out_size, 256),nn.ReLU(),nn.Linear(256, action_dim))
    
#     def _get_conv_out_size(self, shape: tuple[int, int, int]) -> int:
#         with torch.no_grad():
#             dummy_input = torch.zeros(1, *shape)
#             x = F.relu(self.conv1(dummy_input))
#             x = F.relu(self.conv2(x))
#             x = F.relu(self.conv3(x))
#             #print("Conv output shape (batch, channels, height, width):", x.shape)
#             return int(torch.prod(torch.tensor(x.shape[1:])))
#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         x = F.relu(self.conv1(x))
#         x = F.relu(self.conv2(x))
#         x = F.relu(self.conv3(x))
#         # print("Conv output shape before flattening:", x.shape)
#         x = x.view(x.size(0), -1)
#         value = self.value_stream(x)                   # shape: (batch_size, 1)
#         advantage = self.advantage_stream(x)           # shape: (batch_size, action_dim)
#         q_vals = value + (advantage-advantage.mean(dim=1,keepdim=True))  
#         return q_vals




# import torch
# import torch.nn as nn
# import snntorch as snn
# from snntorch import surrogate, functional as SF

# class SNN_DuelingDQN(nn.Module):
#     def __init__(self, input_dim=(4, 80, 80), action_dim=4, hidden_size=1024, num_steps=10):
#         super().__init__()

#         self.num_steps = num_steps
#         beta = 0.9  # leak factor
#         spike_grad = surrogate.fast_sigmoid()

#         self.flattened_size = input_dim[0] * input_dim[1] * input_dim[2]

#         # FC layer instead of CNN
#         self.fc1 = nn.Linear(self.flattened_size, hidden_size)
#         self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True)

#         # Value stream
#         self.value_fc = nn.Linear(hidden_size, 256)
#         self.value_lif = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True)
#         self.value_out = nn.Linear(256, 1)

#         # Advantage stream
#         self.adv_fc = nn.Linear(hidden_size, 256)
#         self.adv_lif = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True)
#         self.adv_out = nn.Linear(256, action_dim)

#     def forward(self, x: torch.Tensor):
#         batch_size = x.shape[0]
#         x = x.view(batch_size, -1)  # flatten
#         x = x / 255.0               # normalize for Poisson

#         mem1 = self.lif1.init_leaky()
#         mem_val = self.value_lif.init_leaky()
#         mem_adv = self.adv_lif.init_leaky()

#         value_spike_sum = 0
#         adv_spike_sum = 0

#         for _ in range(self.num_steps):
#             spike_in = SF.poisson(x)
#             fc1_out = self.fc1(spike_in)
#             spk1, mem1 = self.lif1(fc1_out, mem1)

#             # Value stream
#             val_hidden = self.value_fc(spk1)
#             spk_val, mem_val = self.value_lif(val_hidden, mem_val)
#             value = self.value_out(spk_val)

#             # Advantage stream
#             adv_hidden = self.adv_fc(spk1)
#             spk_adv, mem_adv = self.adv_lif(adv_hidden, mem_adv)
#             advantage = self.adv_out(spk_adv)

#             value_spike_sum += value
#             adv_spike_sum += advantage

#         # Average over time
#         value_avg = value_spike_sum / self.num_steps
#         adv_avg = adv_spike_sum / self.num_steps

#         q_vals = value_avg + (adv_avg - adv_avg.mean(dim=1, keepdim=True))
#         return q_vals
        


In [58]:
def poisson_encoding(input_tensor, num_steps):
    """
    input_tensor: shape (batch_size, channels, height, width)
    Returns: Poisson spike train of shape (num_steps, batch_size, input_size)
    """
    flat_input = input_tensor.view(input_tensor.size(0), -1)  # shape: (batch_size, 25600)
    spike_trains = torch.rand((num_steps, *flat_input.shape), device=flat_input.device) < flat_input.unsqueeze(0)
    return spike_trains.float()


In [59]:
class LeakySurrogate(nn.Module):
  def __init__(self, beta, threshold=1.0):
      super(LeakySurrogate, self).__init__()
      self.beta=beta
      self.threshold=threshold
      self.spike_gradient=self.ATan.apply

  def forward(self,input_,mem):
    spk=self.spike_gradient((mem-self.threshold))  
    reset=(self.beta*spk*self.threshold).detach()  
    mem=self.beta*mem+input_-reset  
    return spk,mem

  @staticmethod
  class ATan(torch.autograd.Function):
      @staticmethod
      def forward(ctx, mem):
          spk=(mem>0).float() 
          ctx.save_for_backward(mem)  
          return spk

      @staticmethod
      def backward(ctx, grad_output):
          (mem,)=ctx.saved_tensors  
          grad=1/(1+(np.pi*mem).pow_(2))*grad_output 
          return grad

In [60]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import Leaky
from snntorch import surrogate, functional as SF

class BreakoutSNN(nn.Module):
    def __init__(self,num_inputs=25600,num_hidden=256,num_outputs=4,beta=0.95):
        super().__init__()
        self.fc1=nn.Linear(num_inputs,num_hidden)
        self.lif1=LeakySurrogate(beta)
        self.fc2=nn.Linear(num_hidden,num_outputs)
        self.lif2=LeakySurrogate(beta)

    def forward(self,x):
        mem1=torch.zeros(x.size(1),self.fc1.out_features,device=x.device)
        mem2=torch.zeros(x.size(1),self.fc2.out_features,device=x.device)

        spk2_rec=[]
        mem2_rec=[]

        for step in range(x.size(0)):
            cur1=self.fc1(x[step])
            spk1,mem1=self.lif1(cur1,mem1)
            cur2=self.fc2(spk1)
            spk2,mem2=self.lif2(cur2,mem2)
            spk2_rec.append(spk2)
            mem2_rec.append(mem2)

        return torch.stack(spk2_rec,dim=0),torch.stack(mem2_rec,dim=0)


def update_target(policy_net, target_net):
    target_net.load_state_dict(policy_net.state_dict())



AGENT IMPLEMENTATION WITH DOUBLE DQN AND PER :

Double DQN Loss Function

In [75]:
import torch.nn.functional as F

def compute_double_dqn_loss(policy_net, target_net, states, actions, rewards, next_states, dones, gamma, is_weights):
    """Compute Double DQN loss with importance sampling"""

    # Get Q-values for current states
    _, mem_out = policy_net(states)       # extract membrane output
    q_values = mem_out[-1]     # [batch_size, action_size]
    actions = actions.long().unsqueeze(1)  # [batch_size, 1]
    q_values_taken = q_values.gather(1, actions).squeeze(1)  # [batch_size]

    # Double DQN: action selection via policy_net, evaluation via target_net
    _, next_mem_policy = policy_net(next_states)
    next_q_values_policy_net = next_mem_policy[-1]
    next_actions = next_q_values_policy_net.argmax(dim=1).unsqueeze(1)

    _, next_mem_target = target_net(next_states)
    next_q_values_target_net = next_mem_target[-1]
    next_q_values = next_q_values_target_net.gather(1, next_actions).squeeze(1)


    # Convert to tensors if needed
    rewards = rewards.to(dtype=torch.float32, device=states.device)
    dones = dones.to(dtype=torch.float32, device=states.device)
     # Convert importance sampling weights to tensor with no grad
    if not torch.is_tensor(is_weights):
        is_weights = torch.tensor(is_weights, dtype=torch.float32, device=states.device)
    else:
      is_weights = is_weights.to(dtype=torch.float32, device=states.device).detach()

    # Compute TD targets
    target_q_values = rewards + gamma * next_q_values * (1 - dones)

    # TD error
    temporal_diff_errors = target_q_values - q_values_taken

    # Compute loss with importance sampling weights
    mse_loss = F.mse_loss(q_values_taken, target_q_values.detach(), reduction='none')  # Detach target
    loss = (is_weights * mse_loss).mean()

    return loss, temporal_diff_errors


Advanced DQN Agent Class

In [76]:
class AdvancedDQNAgent:
    
    def __init__(self, state_shape, action_size, config):
        """Initialize agent with networks and replay buffer"""
        # Store network dimensions and configuration parameters.
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.state_shape = state_shape
        self.action_size = action_size

        self.gamma = config['gamma']
        self.learning_rate = config['learning_rate']
        self.buffer_size = config['buffer_size']
        self.batch_size = config['batch_size']
        self.target_update_freq = config['target_update_freq'] # Frequency (in training steps) to update the target network
        self.initial_replay_size = config['initial_replay_size'] # Minimum number of experiences in the buffer before training starts
        self.alpha = config['alpha']  # For PER prioritization
        self.beta_start = config['beta_start']  # For PER importance sampling
        self.max_episodes = config['max_episodes']
        self.target_score = config['target_score'] # Mean score over 50 episodes

        # Create policy and target networks with identical architectures.
        self.policy_net = BreakoutSNN()
        self.target_net = BreakoutSNN()

        # Initialize the target network with policy network weights.
        self.target_net.load_state_dict(self.policy_net.state_dict())

        # Set up the optimizer and prioritized replay buffer.
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.learning_rate)
        self.replay_buffer = PrioritizedReplayBuffer(self.buffer_size, self.alpha, self.beta_start)

        # Configure exploration parameters for epsilon-greedy strategy.
        self.epsilon_start = 1.0
        self.epsilon_final = 0.01
        self.epsilon_decay = 50000# epsilon will decay exponentially over 50,000 frames
        self.frame_idx = 0 #counter for how many frames the agent has taken so far

    def select_action(self, state: np.ndarray) -> int:
        """Epsilon-greedy action selection"""
        # Calculate current epsilon value using exponential decay schedule.
        self.frame_idx += 1
        self.epsilon = self.epsilon_final + (self.epsilon_start - self.epsilon_final)*\
                        np.exp(-1.0 * self.frame_idx/self.epsilon_decay)

        # Choose between random exploration and greedy exploitation.
        if np.random.random() < self.epsilon :
            return np.random.randint(0, self.action_size)
        else :
            with torch.no_grad():
                state_tensor = torch.from_numpy(state).unsqueeze(0).to(torch.float32).to(self.device)  # shape (1, 4, 80, 80)
                spikes = poisson_encoding(state_tensor, num_steps=10)  # shape (10, 1, 25600)
                _, mem_out = self.policy_net(spikes)  # get membrane potentials
                q_values = mem_out[-1]  # take membrane potential at final timestep
                return q_values.argmax(dim=1).item()

    #Learning and Update methods
    def store_transition(self, state, action, reward, next_state, done):
        """Store experience in replay buffer"""
        # Add the experience tuple to the prioritized replay buffer.
        self.replay_buffer.push(state, action, reward, next_state, done)

    def update(self) -> Optional[float]:
        """Perform learning update"""
        # Check if sufficient experiences are available for training.
        if len(self.replay_buffer) < self.initial_replay_size :
            return None

        # Sample a batch from the prioritized replay buffer.
        experience, indices, is_weights = self.replay_buffer.sample(self.batch_size)

        # Convert experience components to tensors and move to device.
        states, actions, rewards, next_states, dones = zip(*experience)

        states = np.array(states)
        next_states = np.array(next_states)
        actions = np.array(actions)
        rewards = np.array(rewards)
        dones = np.array(dones)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        states_tensor = torch.from_numpy(states).to(self.device)
        next_states_tensor = torch.from_numpy(next_states).to(self.device)
        states_spike = poisson_encoding(states_tensor, num_steps=10)
        next_states_spike = poisson_encoding(next_states_tensor, num_steps=10)
        actions_tensor = torch.from_numpy(actions).to(self.device)
        rewards_tensor = torch.from_numpy(rewards).to(self.device)
        dones_tensor = torch.from_numpy(dones).to(torch.float32)
        dones_tensor = dones_tensor.to(self.device)

        if not torch.is_tensor(is_weights):
          is_weights_tensor = torch.tensor(is_weights, dtype=torch.float32, device=self.device)
        else:
          is_weights_tensor = is_weights.to(dtype=torch.float32, device=self.device).detach()

        # Compute Double DQN loss and TD errors.
        loss, temporal_diff_errors = compute_double_dqn_loss(self.policy_net, self.target_net, states_spike, actions_tensor,
                                                             rewards_tensor, next_states_spike, dones_tensor, self.gamma, is_weights_tensor)

        # Update experience priorities based on TD errors.
        self.replay_buffer.update_priorities(indices, temporal_diff_errors.detach().cpu().numpy())

        # Perform gradient descent optimization step.
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        # return the scalar loss value for logging
        return loss.item()

    def update_target_network(self):
        """Copy weights from policy to target network"""
        # Synchronize target network weights with policy network.
        self.target_net.load_state_dict(self.policy_net.state_dict())

TRAINING LOOP AND EVALUATION ON BREAKOUT :

Training Configuration

In [77]:
config = {
    'learning_rate': 5e-4,  # Learning rate for the optimizer
    'gamma': 0.90,  # Discount factor for future rewards
    'buffer_size': 20000,  # Maximum size of the replay buffer
    'batch_size': 16,  # Number of samples to draw from the buffer for each training step
    'target_update_freq': 300,  # Frequency (in training steps) to update the target network
    'initial_replay_size': 5000,  # Minimum number of experiences in the buffer before training starts
    'alpha': 0.6,  # PER prioritization
    'beta_start': 0.4,  # PER importance sampling
    'max_episodes': 1000,  # Maximum number of episodes to run
    'target_score': 12.0,  # Mean score over 50 episodes
}

Main Training Function Implementation

In [78]:
def train(agent, env, max_episodes=1000, target_update_freq=500):
    all_rewards = []
    losses = []
    total_steps = 0
    episode_rewards = []
    mean_scores = []
    score_window = deque(maxlen=50)
    start_time = time.time()


    for episode in range(1, max_episodes + 1):
        state = env.reset()
        episode_reward = 0
        done = False
        total_reward = 0
        steps = 0
        episode_losses = []


        while not done:
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)
            agent.store_transition(state, action, reward, next_state, done)

            loss = agent.update()

            if loss is not None:
                losses.append(loss)

            state = next_state
            episode_reward += reward
            total_reward += reward
            total_steps += 1

            #if total_steps % target_update_freq == 0:
             #   agent.update_target_network()

        all_rewards.append(episode_reward)

        if episode % 10 == 0:
          avg_reward = np.mean(all_rewards[-10:])
          avg_loss = np.mean(losses[-100:]) if losses else 0
          print(f"Episode {episode} | Avg Reward (10) {avg_reward:.2f} | Avg Loss (100) {avg_loss:.4f} | Epsilon {agent.epsilon:.3f}")

        if agent.frame_idx % config['target_update_freq'] == 0:
         agent.update_target()

        episode_rewards.append(total_reward)
        score_window.append(total_reward)
        mean_score = np.mean(score_window)
        mean_scores.append(mean_score)

        elapsed = time.time() - start_time
        eta = (config['max_episodes'] - episode - 1) * (elapsed / (episode + 1))

        loss_display = f"{loss:.4f}" if loss is not None else "N/A"
        print(f"Episode {episode + 1}/{config['max_episodes']}, "
             f"Reward: {total_reward}, Mean Score: {mean_score:.2f}, "
             f"Epsilon: {getattr(agent,'epsilon',1.0):.3f}, Loss: {loss_display}, "
             f"ETA: {eta / 60:.2f} mins")


        if mean_score >= config['target_score']:
           print(f"Target mean score {config['target_score']} achieved at episode {episode + 1}! Early stopping.")
           break


    env.close()
    return all_rewards, episode_rewards, mean_scores, losses, agent

Visualization and Plotting Function

In [52]:
def plot_training_results1(episode_rewards, mean_scores, losses, save_path="training_summary.png"):
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))

    # --- Plot 1: Episode Rewards with Rolling Mean ---
    axs[0, 0].plot(episode_rewards, label="Episode Reward")
    axs[0, 0].plot(pd.Series(episode_rewards).rolling(50).mean(), label="Rolling Mean (50)", linewidth=2)
    axs[0, 0].set_title("Episode Rewards")
    axs[0, 0].set_xlabel("Episode")
    axs[0, 0].set_ylabel("Reward")
    axs[0, 0].legend()

    # --- Plot 2: Mean Score Over Episodes ---
    axs[0, 1].plot(mean_scores, color='green', label="Mean Score")
    axs[0, 1].axhline(y=12, color='red', linestyle='--', label="Target Score")
    axs[0, 1].set_title("Mean Score Over Last 50 Episodes")
    axs[0, 1].set_xlabel("Episode")
    axs[0, 1].set_ylabel("Mean Score")
    axs[0, 1].legend()

    # --- Plot 3: Loss per Episode ---
    axs[1, 0].plot(losses, color='orange', label="Loss")
    axs[1, 0].plot(pd.Series(losses).rolling(50).mean(), label="Rolling Mean (50)", linewidth=2)
    axs[1, 0].set_title("Training Loss")
    axs[1, 0].set_xlabel("frame")
    axs[1, 0].set_ylabel("Loss")
    axs[1, 0].legend()

    # --- Plot 4: Reward Distribution Histogram ---
    axs[1, 1].hist(episode_rewards, bins=20, color='purple')
    axs[1, 1].set_title("Score Distribution")
    axs[1, 1].set_xlabel("Total Reward")
    axs[1, 1].set_ylabel("Frequency")

    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

Main Execution Block

In [79]:
if __name__ == "__main__":
    print("=== RL Assignment2: Friendly Neural Hood: DQN Agent Training ===")

    # Run training

env = AtariWrapper("BreakoutNoFrameskip-v4")
n_actions = env.env.unwrapped.action_space.n
agent = AdvancedDQNAgent(state_shape=(4, 84, 84), action_size =n_actions, config=config)
all_rewards, episode_rewards, mean_scores, losses, agent = train(agent, env, max_episodes=1000, target_update_freq=config['target_update_freq'])


    # Plot results
plot_training_results1(episode_rewards, mean_scores, losses)

# Condition 1: Check if any mean score is >= 12
condition1 = any(m >= 12 for m in mean_scores)

# Condition 2: Check for any 50-episode block where every reward >= 8
condition2 = any(all(r >= 8 for r in episode_rewards[i:i+50]) for i in range(len(episode_rewards) - 49))

# Final decision
if condition1:
    status = "Phase 01 Complete: Mean score of 12+ achieved"
elif condition2:
    status = "Phase 01 Complete: Found 50-episode block where all scores ≥ 8"
else:
    status = "Training of 1000 episodes completed: Mean score of 12+ achieved/ Found 50-episode block where all scores ≥ 8"

print(f"{status}")


=== RL Assignment2: Friendly Neural Hood: DQN Agent Training ===
Episode 2/1000, Reward: 3.0, Mean Score: 3.00, Epsilon: 0.995, Loss: N/A, ETA: 8.43 mins
Episode 3/1000, Reward: 1.0, Mean Score: 2.00, Epsilon: 0.992, Loss: N/A, ETA: 10.89 mins
Episode 4/1000, Reward: 0.0, Mean Score: 1.33, Epsilon: 0.989, Loss: N/A, ETA: 11.68 mins
Episode 5/1000, Reward: 3.0, Mean Score: 1.75, Epsilon: 0.984, Loss: N/A, ETA: 13.01 mins
Episode 6/1000, Reward: 0.0, Mean Score: 1.40, Epsilon: 0.981, Loss: N/A, ETA: 12.54 mins
Episode 7/1000, Reward: 0.0, Mean Score: 1.17, Epsilon: 0.979, Loss: N/A, ETA: 12.13 mins
Episode 8/1000, Reward: 2.0, Mean Score: 1.29, Epsilon: 0.974, Loss: N/A, ETA: 12.74 mins
Episode 9/1000, Reward: 1.0, Mean Score: 1.25, Epsilon: 0.970, Loss: N/A, ETA: 12.76 mins
Episode 10/1000, Reward: 0.0, Mean Score: 1.11, Epsilon: 0.967, Loss: N/A, ETA: 12.52 mins
Episode 10 | Avg Reward (10) 1.10 | Avg Loss (100) 0.0000 | Epsilon 0.964
Episode 11/1000, Reward: 1.0, Mean Score: 1.10, Eps

C:\Users\SHREEVARDHAN\AppData\Local\Temp\ipykernel_14320\1136278593.py:55: RuntimeWarning: divide by zero encountered in divide
  weights=((1/self.size)*(1/temp))**self._get_beta()
C:\Users\SHREEVARDHAN\AppData\Local\Temp\ipykernel_14320\1136278593.py:56: RuntimeWarning: invalid value encountered in divide
  normalized=weights/weights.max()


TypeError: 'int' object is not iterable